# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZubairQazzi/flyrank-ml-internship-zubair/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions

My lane is **content refresh opportunity scoring**.

Before testing any rule, I inspect the distribution of the main pre-decision signals used in the project:

- past impressions
- past CTR
- past average position
- observed GSC days
- days since the page was last known to be updated

The analysis uses the same March 2026 decision setup as the later modeling work: March 1–21 provides historical signals and March 22–31 is reserved for the later `future_decline` proxy.

The distributions are expected to be uneven and heavy-tailed, especially for impressions and page age. This matters because a small number of very large or very old pages should not determine a rule by themselves.

In [1]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{hf_token}'
    )
    """
)

march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

content_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_content.parquet"
)

signal_query = f"""
WITH daily_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS past_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_clicks
                ELSE 0
            END
        ) AS past_clicks,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
                 AND gsc_data_available IS TRUE
                THEN gsc_sum_position
                ELSE 0
            END
        ) AS past_sum_position,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-21'
              AND gsc_data_available IS TRUE
        ) AS gsc_observed_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ) AS outcome_impressions,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '2026-03-22' AND DATE '2026-03-31'
              AND gsc_data_available IS TRUE
        ) AS outcome_days

    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id, content_hash_id
),

content_dates AS (
    SELECT
        client_hash_id,
        content_hash_id,
        is_published,
        is_deleted,

        NULLIF(
            GREATEST(
                COALESCE(
                    CASE
                        WHEN last_optimized_date <= DATE '2026-03-21'
                        THEN last_optimized_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_updated_date <= DATE '2026-03-21'
                        THEN content_updated_date
                    END,
                    DATE '1900-01-01'
                ),
                COALESCE(
                    CASE
                        WHEN content_created_date <= DATE '2026-03-21'
                        THEN content_created_date
                    END,
                    DATE '1900-01-01'
                )
            ),
            DATE '1900-01-01'
        ) AS last_known_update_date

    FROM read_parquet('{content_path}')
)

SELECT
    d.client_hash_id,
    d.content_hash_id,
    d.past_impressions,
    d.past_clicks,

    ROUND(
        100.0 * d.past_clicks /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_ctr,

    ROUND(
        1.0 * d.past_sum_position /
        NULLIF(d.past_impressions, 0),
        4
    ) AS past_avg_position,

    d.gsc_observed_days,

    DATE_DIFF(
        'day',
        c.last_known_update_date,
        DATE '2026-03-21'
    ) AS days_since_update,

    CASE
        WHEN
            (1.0 * d.outcome_impressions / d.outcome_days)
            <
            0.80 * (
                1.0 * d.past_impressions /
                d.gsc_observed_days
            )
        THEN 1
        ELSE 0
    END AS future_decline

FROM daily_windows d

INNER JOIN content_dates c
    ON d.client_hash_id = c.client_hash_id
   AND d.content_hash_id = c.content_hash_id

WHERE
    c.is_published IS TRUE
    AND COALESCE(c.is_deleted, FALSE) IS FALSE
    AND d.gsc_observed_days >= 7
    AND d.outcome_days >= 5
    AND d.past_impressions >= 100
"""

signal_frame = con.sql(signal_query).df()

signal_frame = (
    signal_frame
    .sort_values(["client_hash_id", "content_hash_id"])
    .reset_index(drop=True)
)

distribution_cols = [
    "past_impressions",
    "past_ctr",
    "past_avg_position",
    "gsc_observed_days",
    "days_since_update",
]

distribution_summary = (
    signal_frame[distribution_cols]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .T
)

print("Rows in signal frame:", len(signal_frame))
print("Unique clients:", signal_frame["client_hash_id"].nunique())
print(
    "Future-decline rate:",
    round(signal_frame["future_decline"].mean(), 4),
)

display(distribution_summary.round(4))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in signal frame: 85967
Unique clients: 39
Future-decline rate: 0.3249


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
past_impressions,85967.0,2049.0768,4689.8230,100.000,263.0000,680.0000,1912.0000,4809.4000,8123.0000,20980.1000,273012.0000
past_ctr,85967.0,0.2789,0.4476,0.000,0.0000,0.1327,0.3982,0.7380,1.0220,1.9737,18.7500
past_avg_position,85967.0,13.2764,13.7450,0.023,4.4814,7.6319,17.2962,31.6838,41.8503,67.6107,115.3731
gsc_observed_days,85967.0,19.9627,2.4879,7.000,20.0000,21.0000,21.0000,21.0000,21.0000,21.0000,21.0000
days_since_update,85967.0,159.9590,136.3216,6.000,26.0000,114.0000,253.0000,380.0000,402.0000,465.0000,484.0000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Rows in signal frame: 85967
Unique clients: 39
Future-decline rate: 0.3249
count	mean	std	min	25%	50%	75%	90%	95%	99%	max
past_impressions	85967.0	2049.0768	4689.8230	100.000	263.0000	680.0000	1912.0000	4809.4000	8123.0000	20980.1000	273012.0000
past_ctr	85967.0	0.2789	0.4476	0.000	0.0000	0.1327	0.3982	0.7380	1.0220	1.9737	18.7500
past_avg_position	85967.0	13.2764	13.7450	0.023	4.4814	7.6319	17.2962	31.6838	41.8503	67.6107	115.3731
gsc_observed_days	85967.0	19.9627	2.4879	7.000	20.0000	21.0000	21.0000	21.0000	21.0000	21.0000	21.0000
days_since_update	85967.0	159.9590	136.3216	6.000	26.0000	114.0000	253.0000	380.0000	402.0000	465.0000	484.0000


In [2]:
def compare_binary_signal(
    frame,
    signal_name,
    condition,
    positive_label,
    reference_label,
):
    temp = frame.copy()

    temp["group"] = np.where(
        condition,
        positive_label,
        reference_label,
    )

    summary = (
        temp
        .groupby("group")["future_decline"]
        .agg(
            rows="count",
            decline_rate="mean",
        )
        .reset_index()
    )

    summary.insert(
        0,
        "signal",
        signal_name,
    )

    return summary


# --------------------------------------------------
# Signal 1: Low CTR
# --------------------------------------------------

ctr_test = compare_binary_signal(
    signal_frame,
    "Low CTR",
    signal_frame["past_ctr"] < 0.20,
    "CTR < 0.20",
    "CTR >= 0.20",
)


# --------------------------------------------------
# Signal 2: Staleness
# --------------------------------------------------

staleness_test = compare_binary_signal(
    signal_frame,
    "Staleness",
    signal_frame["days_since_update"] >= 180,
    "Age >= 180 days",
    "Age < 180 days",
)


# --------------------------------------------------
# Signal 3: Visibility
# --------------------------------------------------

visibility_test = compare_binary_signal(
    signal_frame,
    "Visibility",
    signal_frame["past_impressions"] >= 500,
    "Impressions >= 500",
    "Impressions < 500",
)


signal_tests = pd.concat(
    [
        ctr_test,
        staleness_test,
        visibility_test,
    ],
    ignore_index=True,
)

signal_tests["decline_rate_pct"] = (
    100 * signal_tests["decline_rate"]
)

display(
    signal_tests[
        [
            "signal",
            "group",
            "rows",
            "decline_rate_pct",
        ]
    ].round(2)
)


# --------------------------------------------------
# Difference in decline rate for each hypothesis
# --------------------------------------------------

def rate_difference(
    summary,
    positive_group,
    reference_group,
):
    rates = summary.set_index("group")["decline_rate"]

    return (
        rates[positive_group]
        - rates[reference_group]
    )


differences = pd.DataFrame(
    [
        {
            "signal": "Low CTR",
            "difference_pct_points": 100 * rate_difference(
                ctr_test,
                "CTR < 0.20",
                "CTR >= 0.20",
            ),
        },
        {
            "signal": "Staleness",
            "difference_pct_points": 100 * rate_difference(
                staleness_test,
                "Age >= 180 days",
                "Age < 180 days",
            ),
        },
        {
            "signal": "Visibility",
            "difference_pct_points": 100 * rate_difference(
                visibility_test,
                "Impressions >= 500",
                "Impressions < 500",
            ),
        },
    ]
)

print("\nDifference in observed decline rate:")
display(differences.round(2))

,signal,group,rows,decline_rate_pct
0,Low CTR,CTR < 0.20,49982,37.69
1,Low CTR,CTR >= 0.20,35985,25.26
2,Staleness,Age < 180 days,48550,36.34
3,Staleness,Age >= 180 days,37417,27.49
4,Visibility,Impressions < 500,36077,36.22
5,Visibility,Impressions >= 500,49890,29.79



Difference in observed decline rate:


,signal,difference_pct_points
0,Low CTR,12.44
1,Staleness,-8.84
2,Visibility,-6.43


### Signal verdicts

**Signal 1 — Low CTR: CONFIRMED**

Pages with `past_ctr < 0.20` had a 37.69% observed future-decline rate, compared with 25.26% for pages with CTR at or above 0.20. The difference was +12.44 percentage points.

**Signal 2 — Staleness: OPPOSITE**

Pages at least 180 days old had a 27.49% observed future-decline rate, compared with 36.34% for more recently updated pages. The difference was -8.84 percentage points, so this binary test does not support the assumption that older pages are more likely to decline.

**Signal 3 — Visibility: OPPOSITE**

Pages with at least 500 past impressions had a 29.79% observed future-decline rate, compared with 36.22% for pages below 500 impressions. The difference was -6.43 percentage points. Higher visibility may still increase the value of reviewing a page, but it did not indicate higher decline risk in this slice.

These are observed associations in this dataset and period, not causal effects.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: weak CTR on visible pages

FlyRank's product uses optimization flags such as `needs_ctr_fix` as workflow signals. The actual product flag is not included in this dataset, so I do not attempt to reproduce or predict the private decision rule.

Instead, I test one observable assumption behind a CTR-review workflow: among pages with meaningful existing visibility, do pages with weak historical CTR show a higher observed future-decline rate?

For this test, I restrict the comparison to pages with at least 500 historical impressions and compare pages below versus above a 0.20 historical CTR threshold.

This test does not prove that low CTR causes future decline or that changing a title or snippet will improve performance.

In [3]:
# --------------------------------------------------
# Flag-linked test:
# weak CTR among pages with meaningful visibility
# --------------------------------------------------

visible_pages = signal_frame[
    signal_frame["past_impressions"] >= 500
].copy()

visible_pages["ctr_group"] = np.where(
    visible_pages["past_ctr"] < 0.20,
    "CTR < 0.20",
    "CTR >= 0.20",
)

flag_linked_test = (
    visible_pages
    .groupby("ctr_group")["future_decline"]
    .agg(
        rows="count",
        decline_rate="mean",
    )
    .reset_index()
)

flag_linked_test["decline_rate_pct"] = (
    100 * flag_linked_test["decline_rate"]
)

display(
    flag_linked_test[
        [
            "ctr_group",
            "rows",
            "decline_rate_pct",
        ]
    ].round(2)
)

rates = (
    flag_linked_test
    .set_index("ctr_group")["decline_rate"]
)

difference_pp = 100 * (
    rates["CTR < 0.20"]
    - rates["CTR >= 0.20"]
)

print(
    "Low-CTR minus higher-CTR decline rate:",
    round(difference_pp, 2),
    "percentage points",
)

,ctr_group,rows,decline_rate_pct
0,CTR < 0.20,25920,37.43
1,CTR >= 0.20,23970,21.54


Low-CTR minus higher-CTR decline rate: 15.89 percentage points


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Practical takeaway

The signal audit shows that the tested features should not all be interpreted the same way.

Low historical CTR was the clearest observed warning signal. Pages below a 0.20 CTR threshold had a higher future-decline rate overall, and the difference became even larger when the test was restricted to pages with at least 500 historical impressions.

Staleness and visibility did not behave like simple risk predictors. Older pages were not consistently more likely to decline, and higher-visibility pages actually showed a lower observed decline rate in this slice.

For a content team, this means visibility and page age are better used to prioritize the potential value of a review, while weak CTR can help identify pages worth investigating. None of these signals should trigger an automatic refresh without human review.

In [4]:
practical_summary = pd.DataFrame(
    [
        {
            "signal": "Low CTR",
            "verdict": "CONFIRMED",
            "observed_result": (
                "CTR < 0.20 had a 12.44 percentage-point "
                "higher decline rate overall."
            ),
            "practical_use": (
                "Use as a review signal, especially on visible pages."
            ),
        },
        {
            "signal": "Staleness",
            "verdict": "OPPOSITE",
            "observed_result": (
                "Pages >= 180 days old had an 8.84 percentage-point "
                "lower decline rate in the binary test."
            ),
            "practical_use": (
                "Use age for review priority/context, not as proof of risk."
            ),
        },
        {
            "signal": "Visibility",
            "verdict": "OPPOSITE",
            "observed_result": (
                "Pages with >= 500 impressions had a 6.43 percentage-point "
                "lower decline rate."
            ),
            "practical_use": (
                "Use visibility to estimate review value, not decline probability."
            ),
        },
        {
            "signal": "Low CTR on visible pages",
            "verdict": "CONFIRMED",
            "observed_result": (
                "Low-CTR visible pages had a 15.89 percentage-point "
                "higher observed decline rate."
            ),
            "practical_use": (
                "Prioritize human investigation of click capture and intent match."
            ),
        },
    ]
)

print("Signal audit summary:")
display(practical_summary)

Signal audit summary:


,signal,verdict,observed_result,practical_use
0,Low CTR,CONFIRMED,CTR < 0.20 had a 12.44 percentage-point higher...,"Use as a review signal, especially on visible ..."
1,Staleness,OPPOSITE,Pages >= 180 days old had an 8.84 percentage-p...,"Use age for review priority/context, not as pr..."
2,Visibility,OPPOSITE,Pages with >= 500 impressions had a 6.43 perce...,"Use visibility to estimate review value, not d..."
3,Low CTR on visible pages,CONFIRMED,Low-CTR visible pages had a 15.89 percentage-p...,Prioritize human investigation of click captur...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.